In [4]:
import pandas as pd
import os
import warnings
import librosa
import soundfile as sf  # To save the trimmed wav files
from tqdm import tqdm

# --- 1. CONFIGURATION ---
warnings.filterwarnings("ignore")

csv_files = [
    "master_valid_dataset.csv",
    "master_test_known_dataset.csv",
    "master_test_unknown_dataset.csv"
]

# --- 2. THE STANDARDIZATION + VAD FUNCTION ---
def standardize_audio_with_vad(csv_path):
    print(f"\n🚀 Starting Conversion + VAD: {csv_path}")
    df = pd.read_csv(csv_path)
    new_paths = []
    error_log = []
    
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Standardizing + Trimming"):
        old_path = row['file_path']
        
        # Folder logic
        audio_dir = os.path.dirname(old_path)
        new_dir = os.path.join(audio_dir, "wav_16k_trimmed") # New name to distinguish
        os.makedirs(new_dir, exist_ok=True)
        
        new_filename = row['file_id'].rsplit('.', 1)[0] + ".wav"
        new_path = os.path.join(new_dir, new_filename)
        
        success = True
        if not os.path.exists(new_path):
            try:
                # A. LOAD AUDIO (Phase 2 core)
                # Librosa automatically handles conversion to mono and 16kHz
                y, sr = librosa.load(old_path, sr=16000, mono=True)

                # B. VOICE ACTIVITY DETECTION / TRIMMING (The New Feature)
                # top_db=20: Anything quieter than 20dB below max is considered silence
                # This removes leading/trailing "dead air"
                y_trimmed, _ = librosa.effects.trim(y, top_db=20)

                y_norm = librosa.util.normalize(y_trimmed)

                # C. SAVE TO DISK
                sf.write(new_path, y_norm, sr)
                
            except Exception as e:
                error_log.append(f"Error processing {old_path}: {e}")
                success = False
        
        if success:
            new_paths.append(os.path.abspath(new_path))
        else:
            new_paths.append(old_path)
    
    # Save the updated CSV
    df['file_path'] = new_paths
    output_name = csv_path.replace(".csv", "_standardized2.csv")
    df.to_csv(output_name, index=False, encoding="utf-8-sig")
    
    print(f"✅ Finished: {output_name}")
    return error_log

# --- 3. EXECUTION ---
for csv in csv_files:
    if os.path.exists(csv):
        errors = standardize_audio_with_vad(csv)

print("\n🎯 Silence removed. Data is now high-density and ready for HPC upload!")


🚀 Starting Conversion + VAD: master_valid_dataset.csv


Standardizing + Trimming: 100%|█████████████████████████████████████████████████| 32176/32176 [00:03<00:00, 9945.25it/s]


✅ Finished: master_valid_dataset_standardized2.csv

🚀 Starting Conversion + VAD: master_test_known_dataset.csv


Standardizing + Trimming: 100%|███████████████████████████████████████████████████| 31740/31740 [42:07<00:00, 12.56it/s]


✅ Finished: master_test_known_dataset_standardized2.csv

🚀 Starting Conversion + VAD: master_test_unknown_dataset.csv


Standardizing + Trimming: 100%|███████████████████████████████████████████████████| 19672/19672 [32:00<00:00, 10.24it/s]


✅ Finished: master_test_unknown_dataset_standardized2.csv

🎯 Silence removed. Data is now high-density and ready for HPC upload!
